# O–H vibrational model

Cleaned and organized for reproducible execution from the repository root.


##### Simulate O–H vibrational mode coupled to a two-level electronic system with energy coupling, transfer, and dissipation using QuTiP

Pseudocode:
1. Import libraries (numpy, matplotlib, qutip)
2. Define physical parameters: electronic splitting, vibrational frequency, coupling strengths, truncation N, dissipation rates
3. Construct operators: electronic qeye, σ_z, σ_±; vibrational destroy a
4. Tensor operators to composite space
5. Build Hamiltonian: H_e + H_v + Holstein-type vibronic + exchange-type coupling
6. Choose initial state: electronic excited, vibrational ground
7. Define time grid
8. Define collapse operators for electronic relaxation and vibrational damping
9. Solve master equation (mesolve) to get ⟨σ_z⟩ and vibrational occupancy
10. Plot dynamics
'''

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qutip import (
    tensor, qeye, sigmaz, sigmap, sigmam,
    destroy, basis, mesolve
)

In [ ]:
# 1. Physical parameters
delta_e    = 1.0    # electronic splitting frequency
omega_v    = 1.2    # O–H vibrational frequency
N_vib      = 30     # vibrational mode truncation

g_holstein = 0.05   # Holstein-type coupling σ_z (a + a†)
g_ex       = 0.02   # energy-exchange coupling σ_+ a + σ_- a†

gamma_e    = 0.01   # electronic relaxation rate
kappa_v    = 0.02   # vibrational damping rate

In [ ]:
# 2. Microwave drive parameters
drive_amp  = 2    # drive amplitude
drive_freq = 1.2    # drive frequency (resonant with vib mode)

In [ ]:
# 3. Operators
# electronic two-level system
i_e  = qeye(2)
Sz   = sigmaz()
Sp   = sigmap()
Sm   = sigmam()
# vibrational mode
i_v  = qeye(N_vib)
a    = destroy(N_vib)
a_d  = a.dag()

# Tensor operators
i_e_nv = tensor(i_e, a_d * a)
Sz_I   = tensor(Sz, i_v)
Sp_a   = tensor(Sp, a)
Sm_ad  = tensor(Sm, a_d)

In [ ]:
# 4. Static Hamiltonian
H_e   = (delta_e / 2) * Sz_I
H_v   = omega_v * i_e_nv
H_hz  = g_holstein * tensor(Sz, a + a_d)
H_ex  = g_ex       * (Sp_a + Sm_ad)
H_stat = H_e + H_v + H_hz + H_ex

In [ ]:
# 5. Drive Hamiltonian (vibrational drive)
H_drive = tensor(i_e, a + a_d)
# time-dependent coefficient f(t) = drive_amp * cos(drive_freq * t)

In [ ]:
# 6. Initial state and time grid
psi0  = tensor(basis(2,1), basis(N_vib,0))  # electronic excited, vib ground
tlist = np.linspace(0, 50, 500)

In [ ]:
# 7. Collapse operators
c_e = np.sqrt(gamma_e) * tensor(Sm, i_v)  # electronic relaxation
c_v = np.sqrt(kappa_v) * tensor(i_e, a)   # vibrational damping
c_ops = [c_e, c_v]

In [ ]:
# 8. Time-dependent Hamiltonian list for mesolve\# H = H_stat + drive_amp*cos(ω_d t)*H_drive
H_td = [H_stat, [H_drive, lambda t, args: drive_amp * np.cos(drive_freq * t)]]

In [ ]:
# 9. Solve master equation
obs = [Sz_I, i_e_nv]
result = mesolve(H_td, psi0, tlist, c_ops, obs)

In [ ]:
# 10. Plot results
plt.figure()
plt.plot(tlist, result.expect[0])
plt.xlabel('Time')
plt.ylabel(r'$\langle \sigma_z \rangle$')
plt.title('Electronic dynamics with microwave drive')

plt.figure()
plt.plot(tlist, result.expect[1])
plt.xlabel('Time')
plt.ylabel('Vibrational ⟨n⟩')
plt.title('Vibrational occupancy with drive')

if __name__ == '__main__':
    plt.show()